A GLM (Generalized Linear Model) is a flexible extension of linear regression that 
allows modelling of non‑normal data such as claim counts and claim costs. GLMs combine 
a probability distribution (e.g., Poisson or Gamma) with a link function (typically a 
log link) to ensure predictions are valid and to allow multiplicative effects of 
rating factors. This framework is widely used in insurance pricing because it handles 
skewed data, categorical variables, and exposure adjustments in a statistically 
sound way.

In [4]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

data = pd.read_csv("../data/processed/prepared_data.csv")


## Frequency GLM
A Poisson GLM is used for modelling claim frequency because the response variable 
(ClaimNb) is a count. The Poisson distribution assumes that the mean and variance are 
equal, which is often too restrictive for insurance data. A quasi-Poisson model 
relaxes this assumption by allowing the variance to exceed the mean, making it more 
appropriate when overdispersion is present.

A log link function is used to ensure that the predicted claim frequency is always 
positive and to allow multiplicative effects of rating factors. Under the log link, 
each coefficient represents a percentage change in expected frequency relative to the 
baseline category.


In [7]:
freq_model = smf.glm(
    formula="ClaimNb ~ DriverAge + Power + Region + Gas + Density",
    data=data,
    family=sm.families.Poisson(),
    offset=np.log(data["Exposure"])
).fit()

print(freq_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               413960
Model:                            GLM   Df Residuals:                   413936
Model Family:                 Poisson   Df Model:                           23
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -73766.
Date:                Sat, 21 Mar 2026   Deviance:                   1.1418e+05
Time:                        14:26:09   Pearson chi2:                 7.54e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.002429
Covariance Type:            nonrobust                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

The frequency GLM shows that several rating factors significantly influence the 
expected number of claims. Higher power vehicles tend to have higher claim frequency, 
regular gas vehicles have lower frequency than the baseline fuel type, and frequency 
decreases with driver age. Density has a positive effect, indicating that drivers in 
more densely populated areas experience more claims. The large Pearson chi-square 
value indicates overdispersion, so a quasi-Poisson model is more appropriate.